<a href="https://colab.research.google.com/github/Shineii86/MoeStickerBot/blob/main/notebooks/MoeStickerBotV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <img src="https://capsule-render.vercel.app/api?type=waving&height=300&color=gradient&text=𝗠𝗼𝗲%20𝗦𝘁𝗶𝗰𝗸𝗲𝗿%20𝗕𝗼𝘁&fontAlignY=30&fontSize=100&desc=𝖢𝗈𝗅𝖺𝖻%20𝖤𝖽𝗂𝗍𝗂𝗈𝗇%20—%20𝖲𝖾𝗅𝖿‑𝖧𝗈𝗌𝗍%20𝖸𝗈𝗎𝗋%20𝖳𝖾𝗅𝖾𝗀𝗋𝖺𝗆%20𝖲𝗍𝗂𝖼𝗄𝖾𝗋%20𝖡𝗈𝗍&descSize=30" alt="Moe Sticker Bot">
  <p><b>Import LINE/Kakao · Create · Manage — all in one notebook</b></p>
</div>

---

In [ ]:
#@title 📦 1. Setup Environment & Build Bot (Run Once)

import sys, time, subprocess, os, urllib.request, json, requests
from itertools import cycle
from tqdm.notebook import tqdm

# ========== ANSI COLORS ==========
class C:
    R = '\033[0m'; B = '\033[1m'; D = '\033[2m'
    BLK = '\033[30m'; RD = '\033[31m'; GN = '\033[32m'; YL = '\033[33m'
    BL = '\033[34m'; MG = '\033[35m'; CY = '\033[36m'; WH = '\033[37m'
    BRD = '\033[91m'; BGN = '\033[92m'; BYL = '\033[93m'; BBL = '\033[94m'
    BMG = '\033[95m'; BCY = '\033[96m'; BWH = '\033[97m'
    BGRD = '\033[41m'; BGGN = '\033[42m'; BGYL = '\033[43m'; BGBL = '\033[44m'

def success(m): print(f"{C.B}{C.BGGN}{C.BLK} ✓ {m} {C.R}")
def error(m): print(f"{C.B}{C.BGRD}{C.WH} ✗ {m} {C.R}")
def info(m): print(f"{C.B}{C.BGBL}{C.WH} ℹ {m} {C.R}")
def warn(m): print(f"{C.B}{C.BGYL}{C.BLK} ⚠ {m} {C.R}")
def header(t): print(f"\n{C.B}{C.BCY}{'═'*50}\n  {t}\n{'═'*50}{C.R}\n")
def spinner(msg, dur=2):
    frames = cycle(['⠋','⠙','⠹','⠸','⠼','⠴','⠦','⠧','⠇','⠏'])
    end = time.time() + dur
    while time.time() < end:
        sys.stdout.write(f'\r{C.BCY}{next(frames)} {msg}{C.R}  ')
        sys.stdout.flush()
        time.sleep(0.1)
    sys.stdout.write(f'\r{C.BGN}✔{C.R} {msg}   \n')

print(f"{C.B}{C.BGN}✨ Ready!{C.R}")

# ========== INSTALL DEPENDENCIES ==========
header("Installing System Dependencies")
spinner("Updating packages", 1)
!apt-get update -qq 2>/dev/null
!apt-get install -y -qq imagemagick libarchive-tools ffmpeg curl gifsicle python3 exiv2 2>/dev/null
success("Core packages installed")

# Go
url = "https://go.dev/dl/go1.21.5.linux-amd64.tar.gz"
print(f"{C.CY}⬇ Downloading Go...{C.R}")
with tqdm(unit='B', unit_scale=True, desc=f"{C.BCY}Go{C.R}") as t:
    urllib.request.urlretrieve(url, "go.tar.gz", reporthook=lambda b,bs,total: t.update(b*bs-t.n))
!tar -C /usr/local -xzf go.tar.gz
os.environ['PATH'] += ":/usr/local/go/bin"
os.environ['GOPATH'] = "/root/go"
os.environ['GO111MODULE'] = "on"
!mkdir -p $GOPATH
success(f"Go {subprocess.getoutput('go version').split()[2]} installed")

# ========== PYTHON HELPERS ==========
header("Installing Python Helpers")
helpers = [("msb_emoji.py","Emoji"), ("msb_kakao_decrypt.py","Kakao"), ("msb_rlottie.py","Lottie")]
for f,desc in helpers:
    !wget -q https://raw.githubusercontent.com/Shineii86/MoeStickersBot/master/tools/{f} -O /usr/local/bin/{f}
    !chmod +x /usr/local/bin/{f}
    print(f"  {C.GN}✓{C.R} {desc}")
success("Helpers installed")

# ========== BUILD BOT ==========
header("Building MoeStickersBot")
!rm -rf MoeStickersBot
!git clone --depth 1 https://github.com/Shineii86/MoeStickersBot.git 2>&1 | grep -v "Cloning"
%cd MoeStickersBot
spinner("Downloading Go modules", 2)
!go mod download
spinner("Compiling binary", 3)
!go build -o MoeStickersBot cmd/MoeStickersBot/main.go
if os.path.exists("MoeStickersBot"):
    sz = os.path.getsize("MoeStickersBot")/1024/1024
    success(f"Build complete — Binary: {sz:.1f} MB")
else:
    error("Build failed")


In [ ]:
#@title ⚙️ 2. Configure & Launch Bot

# ========== CONFIGURATION ==========
BOT_TOKEN = ""  #@param {type:"string"}
ENABLE_DB = False  #@param {type:"boolean"}
DB_ADDR = "localhost:3306"  #@param {type:"string"}
DB_USER = "moe_bot"  #@param {type:"string"}
DB_PASS = ""  #@param {type:"string"}
DB_NAME = "moe_sticker_bot"  #@param {type:"string"}
ENABLE_WEBAPP = False  #@param {type:"boolean"}
WEBAPP_PORT = 8080  #@param {type:"integer"}
NGROK_AUTHTOKEN = ""  #@param {type:"string"}
DATA_DIR = "moe_sticker_bot_data"  #@param {type:"string"}
LOG_LEVEL = "info"  #@param ["debug", "info", "warn", "error"]
HTTP_PROXY = ""  #@param {type:"string"}

header("Configuration")
if BOT_TOKEN:
    print(f"  {C.GN}✓{C.R} BOT_TOKEN = {BOT_TOKEN[:8]}...{BOT_TOKEN[-4:]}")
else:
    warn("BOT_TOKEN is missing!")

if ENABLE_WEBAPP and not NGROK_AUTHTOKEN:
    warn("WebApp enabled but no ngrok token — disabled")
    ENABLE_WEBAPP = False

# ========== NGROK (if enabled) ==========
WEBAPP_URL = ""
if ENABLE_WEBAPP:
    header("Setting up ngrok Tunnel")
    if not os.path.exists("./ngrok"):
        !wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz && tar -xzf ngrok*.tgz && chmod +x ngrok
    !./ngrok config add-authtoken {NGROK_AUTHTOKEN}
    !pkill -f ngrok || true
    ngrok_proc = subprocess.Popen(["./ngrok", "http", str(WEBAPP_PORT), "--log", "stdout"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    spinner("Starting ngrok", 3)
    for _ in range(10):
        try:
            r = requests.get("http://127.0.0.1:4040/api/tunnels")
            if r.status_code==200:
                tuns = r.json()['tunnels']
                if tuns:
                    WEBAPP_URL = tuns[0]['public_url']
                    success(f"ngrok URL: {WEBAPP_URL}")
                    break
        except: pass
        time.sleep(1)
    else:
        error("Could not retrieve ngrok URL")
        ENABLE_WEBAPP = False

# ========== LAUNCH BOT ==========
header("Launching Bot")
if not BOT_TOKEN:
    error("No BOT_TOKEN provided. Aborting.")
    sys.exit(1)

cmd_line = ["./MoeStickersBot", f"--bot_token={BOT_TOKEN}", f"--log_level={LOG_LEVEL}", f"--data_dir={DATA_DIR}"]
if ENABLE_DB and DB_ADDR:
    cmd_line.extend([f"--db_addr={DB_ADDR}", f"--db_user={DB_USER}", f"--db_pass={DB_PASS}", f"--db_name={DB_NAME}"])
if ENABLE_WEBAPP and WEBAPP_URL:
    cmd_line.append(f"--webapp_url={WEBAPP_URL}")
    cmd_line.append(f"--webapp_listen_addr=0.0.0.0:{WEBAPP_PORT}")
if HTTP_PROXY:
    cmd_line.append(f"--http_proxy={HTTP_PROXY}")

print(f"{C.D}Command: {' '.join(cmd_line).replace(BOT_TOKEN, '[REDACTED]')}{C.R}")
log_out = open("bot_stdout.log", "w")
log_err = open("bot_stderr.log", "w")
process = subprocess.Popen(cmd_line, stdout=log_out, stderr=log_err)
spinner("Starting bot", 3)
time.sleep(2)

if process.poll() is None:
    success(f"Bot is RUNNING — PID {process.pid}")
    print(f"{C.B}{C.BGN}📱 Send /start to your bot on Telegram!{C.R}")
    if WEBAPP_URL:
        print(f"{C.B}{C.BCY}🌐 WebApp: {WEBAPP_URL}{C.R}")
else:
    error("Bot exited immediately. Check logs:")
    !cat bot_stderr.log


In [ ]:
#@title 📜 3. Monitor & Control

ACTION = "View Logs"  #@param ["View Logs", "Stop Bot"]
LOG_TYPE = "stderr"  #@param ["stdout", "stderr"]
LINES = 30  #@param {type:"slider", min:10, max:100, step:10}

if ACTION == "View Logs":
    print(f"{C.BCY}📄 Last {LINES} lines of bot_{LOG_TYPE}.log:{C.R}\n")
    !tail -n {LINES} bot_{LOG_TYPE}.log
else:
    header("Shutdown")
    !pkill -f MoeStickersBot && print(f"{C.BGRD}Bot terminated{C.R}") || print(f"{C.YL}No bot running{C.R}")
    !pkill -f ngrok && print(f"{C.BGRD}ngrok terminated{C.R}") || print(f"{C.YL}No ngrok running{C.R}")
    success("Cleanup complete")


---
<div align="center">
  <img src="https://capsule-render.vercel.app/api?type=waving&color=gradient&customColorList=12,14,20,24,27&height=100&section=footer" width="100%">
  <p>Made with ❤️ for the sticker community</p>
</div>